In [ ]:
# In[1]:

import pandas as pd
import numpy as np

# Load files and parse timestamps as UTC
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_logs_df = pd.read_csv("error_logs.csv")

for df in (metric_df, trace_df, log_df, error_logs_df):
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# Aggregate helper to compute count, P95, min/max timestamp
def aggregate_time_series(df, group_cols, value_col_name):
    grouped = (
        df.groupby(group_cols)
          .agg(
              count=(value_col_name, 'size'),
              p95=(value_col_name, lambda x: np.nan if x.dropna().shape[0]==0 else float(np.percentile(x.dropna(), 95))),
              min_ts=('timestamp', 'min'),
              max_ts=('timestamp', 'max')
          )
          .reset_index()
    )
    # Limit numeric precision of p95 to 6 decimals
    grouped['p95'] = grouped['p95'].round(6)
    return grouped

# Metrics: group by (cmdb_id, kpi_name)
metrics_aggr = aggregate_time_series(metric_df, ['cmdb_id', 'kpi_name'], 'value')
metrics_summary = metrics_aggr.sort_values('count', ascending=False).head(20)

# Traces: group by (cmdb_id, trace_name)
traces_aggr = aggregate_time_series(trace_df, ['cmdb_id', 'trace_name'], 'value')
traces_summary = traces_aggr.sort_values('count', ascending=False).head(20)

# Logs: group by (cmdb_id, log_name)
logs_aggr = aggregate_time_series(log_df, ['cmdb_id', 'log_name'], 'value')
logs_summary = logs_aggr.sort_values('count', ascending=False).head(20)

# Unique cmdb_id values across metric, trace, log
unique_cmdb_ids = sorted(
    set(metric_df['cmdb_id'].dropna().unique())
    .union(set(trace_df['cmdb_id'].dropna().unique()))
    .union(set(log_df['cmdb_id'].dropna().unique()))
)

# Error logs summary: row count and earliest/latest timestamps
error_count = len(error_logs_df)
if error_count > 0:
    error_earliest = error_logs_df['timestamp'].min()
    error_latest = error_logs_df['timestamp'].max()
else:
    error_earliest = pd.NaT
    error_latest = pd.NaT

error_logs_summary = pd.DataFrame({
    'row_count': [error_count],
    'earliest_timestamp_utc': [error_earliest],
    'latest_timestamp_utc': [error_latest]
})

# Display compact results (DataFrames and small summaries)
metrics_summary, traces_summary, logs_summary, unique_cmdb_ids, error_logs_summary

```
Out[1]:
```
```python
# Compose a concise plain-English summary based on previously computed summaries
top_metrics = metrics_summary[['cmdb_id','kpi_name','count','p95']].head(3).to_dict('records')
top_traces = traces_summary[['cmdb_id','trace_name','count','p95']].head(3).to_dict('records')
top_logs = logs_summary[['cmdb_id','log_name','count','p95']].head(3).to_dict('records')

unique_count = len(unique_cmdb_ids)
sample_ids = unique_cmdb_ids[:8]

error_count = int(error_logs_summary.at[0, 'row_count'])
error_earliest = error_logs_summary.at[0, 'earliest_timestamp_utc']
error_latest = error_logs_summary.at[0, 'latest_timestamp_utc']

lines = []
lines.append("Summary of telemetry aggregates (concise):")
lines.append("")
lines.append("Top metric series by data point count (top 3):")
for r in top_metrics:
    lines.append(f"- {r['cmdb_id']} | {r['kpi_name']} — count={r['count']}, P95={r['p95']:.6f}")
lines.append("")
lines.append("Top trace series by data point count (top 3):")
for r in top_traces:
    lines.append(f"- {r['cmdb_id']} | {r['trace_name']} — count={r['count']}, P95={r['p95']:.6f}")
lines.append("")
lines.append("Top log series by data point count (top 3):")
for r in top_logs:
    lines.append(f"- {r['cmdb_id']} | {r['log_name']} — count={r['count']}, P95={r['p95']:.6f}")
lines.append("")
lines.append(f"Unique cmdb_id count across metric/trace/log: {unique_count} (examples: {', '.join(sample_ids)})")
lines.append("")
lines.append(f"Error logs: total rows = {error_count}. Earliest timestamp = {error_earliest}, latest = {error_latest}.")
lines.append("")
lines.append("Interpretation notes: many series have 25 data points in the shown window; logs show high row_count P95 for ts-travel2-service and ts-travel-service; metrics include socket and resource KPIs for several ts-admin-* services and ts-voucher-service.")

summary_text = "\n".join(lines)
summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

(                         cmdb_id    kpi_name  count           p95                    min_ts                    max_ts
342           ts-voucher-service      socket     25  4.000000e+00 2024-01-24 03:20:00+00:00 2024-01-24 03:44:00+00:00
0    ts-admin-basic-info-service         cpu     25  1.206891e+00 2024-01-24 03:20:00+00:00 2024-01-24 03:44:00+00:00
1    ts-admin-basic-info-service      diskio     25  5.688653e+04 2024-01-24 03:20:00+00:00 2024-01-24 03:44:00+00:00
2    ts-admin-basic-info-service  latency-50     25  1.811100e-02 2024-01-24 03:20:00+00:00 2024-01-24 03:44:00+00:00
3    ts-admin-basic-info-service  latency-90     25  2.785800e-02 2024-01-24 03:20:00+00:00 2024-01-24 03:44:00+00:00
4    ts-admin-basic-info-service         mem     25  2.298079e+08 2024-01-24 03:20:00+00:00 2024-01-24 03:44:00+00:00
5    ts-admin-basic-info-service      socket     25  7.026667e+00 2024-01-24 03:20:00+00:00 2024-01-24 03:44:00+00:00
6    ts-admin-basic-info-service    workload     25  7.233100e-01 2024-01-24 03:20:00+00:00 2024-01-24 03:44:00+00:00
7         ts-admin-order-service         cpu     25  1.341080e-01 2024-01-24 03:20:00+00:00 2024-01-24 03:44:00+00:00
8         ts-admin-order-service      diskio     25  6.137395e+04 2024-01-24 03:20:00+00:00 2024-01-24 03:44:00+00:00
9         ts-admin-order-service         mem     25  1.931765e+08 2024-01-24 03:20:00+00:00 2024-01-24 03:44:00+00:00
10        ts-admin-order-service      socket     25  3.402424e+00 2024-01-24 03:20:00+00:00 2024-01-24 03:44:00+00:00
11        ts-admin-route-service         cpu     25  1.309050e-01 2024-01-24 03:20:00+00:00 2024-01-24 03:44:00+00:00
12        ts-admin-route-service      diskio     25  5.063827e+04 2024-01-24 03:20:00+00:00 2024-01-24 03:44:00+00:00
13        ts-admin-route-service         mem     25  1.950181e+08 2024-01-24 03:20:00+00:00 2024-01-24 03:44:00+00:00
14        ts-admin-route-service      socket     25  3.356667e+00 2024-01-24 03:20:00+00:00 2024-01-24 03:44:00+00:00
15       ts-admin-travel-service         cpu     25  1.271138e+00 2024-01-24 03:20:00+00:00 2024-01-24 03:44:00+00:00
16       ts-admin-travel-service      diskio     25  5.947511e+04 2024-01-24 03:20:00+00:00 2024-01-24 03:44:00+00:00
17       ts-admin-travel-service       error     25  0.000000e+00 2024-01-24 03:20:00+00:00 2024-01-24 03:44:00+00:00
18       ts-admin-travel-service  latency-50     25  2.372510e-01 2024-01-24 03:20:00+00:00 2024-01-24 03:44:00+00:00,                 cmdb_id                                         trace_name  count         p95                    min_ts                    max_ts
399  ts-travel2-service              trace.to_ts-travel2-service.row_count     25  640.200000 2024-01-24 03:20:00+00:00 2024-01-24 03:44:00+00:00
0                  root  trace.to_ts-admin-basic-info-service.duration_...     25    0.020453 2024-01-24 03:20:00+00:00 2024-01-24 03:44:00+00:00
1                  root  trace.to_ts-admin-basic-info-service.duration_p95     25    0.030820 2024-01-24 03:20:00+00:00 2024-01-24 03:44:00+00:00
2                  root    trace.to_ts-admin-basic-info-service.error_rate     25    0.000000 2024-01-24 03:20:00+00:00 2024-01-24 03:44:00+00:00
3                  root     trace.to_ts-admin-basic-info-service.row_count     25   21.000000 2024-01-24 03:20:00+00:00 2024-01-24 03:44:00+00:00
4                  root     trace.to_ts-admin-travel-service.duration_mean     25    0.274613 2024-01-24 03:20:00+00:00 2024-01-24 03:44:00+00:00
5                  root      trace.to_ts-admin-travel-service.duration_p95     25    0.452143 2024-01-24 03:20:00+00:00 2024-01-24 03:44:00+00:00
6                  root        trace.to_ts-admin-travel-service.error_rate     25    0.000000 2024-01-24 03:20:00+00:00 2024-01-24 03:44:00+00:00
7                  root         trace.to_ts-admin-travel-service.row_count     25   10.800000 2024-01-24 03:20:00+00:00 2024-01-24 03:44:00+00:00
398  ts-travel2-service             trace.to_ts-travel2-service.error_rate     25    0.000000 2024-01-24 03:20:00+00:00 2024-01-24 03:44:00+00:00
397  ts-travel2-service           trace.to_ts-travel2-service.duration_p95     25    0.018988 2024-01-24 03:20:00+00:00 2024-01-24 03:44:00+00:00
396  ts-travel2-service          trace.to_ts-travel2-service.duration_mean     25    0.008608 2024-01-24 03:20:00+00:00 2024-01-24 03:44:00+00:00
356   ts-travel-service            trace.to_ts-route-service.duration_mean     25    0.004347 2024-01-24 03:20:00+00:00 2024-01-24 03:44:00+00:00
357   ts-travel-service             trace.to_ts-route-service.duration_p95     25    0.004874 2024-01-24 03:20:00+00:00 2024-01-24 03:44:00+00:00
358   ts-travel-service               trace.to_ts-route-service.error_rate     25    0.000000 2024-01-24 03:20:00+00:00 2024-01-24 03:44:00+00:00
359   ts-travel-service                trace.to_ts-route-service.row_count     25  142.600000 2024-01-24 03:20:00+00:00 2024-01-24 03:44:00+00:00
360   ts-travel-service       trace.to_ts-ticketinfo-service.duration_mean     25    0.015173 2024-01-24 03:20:00+00:00 2024-01-24 03:44:00+00:00
361   ts-travel-service        trace.to_ts-ticketinfo-service.duration_p95     25    0.020245 2024-01-24 03:20:00+00:00 2024-01-24 03:44:00+00:00
362   ts-travel-service          trace.to_ts-ticketinfo-service.error_rate     25    0.000000 2024-01-24 03:20:00+00:00 2024-01-24 03:44:00+00:00
363   ts-travel-service           trace.to_ts-ticketinfo-service.row_count     25   51.600000 2024-01-24 03:20:00+00:00 2024-01-24 03:44:00+00:00,                         cmdb_id         log_name  count    p95                    min_ts                    max_ts
0   ts-admin-basic-info-service  log.error_count     25    0.0 2024-01-24 03:20:00+00:00 2024-01-24 03:44:00+00:00
1   ts-admin-basic-info-service    log.row_count     25   31.8 2024-01-24 03:20:00+00:00 2024-01-24 03:44:00+00:00
2       ts-admin-travel-service  log.error_count     25    0.0 2024-01-24 03:20:00+00:00 2024-01-24 03:44:00+00:00
3       ts-admin-travel-service    log.row_count     25   32.4 2024-01-24 03:20:00+00:00 2024-01-24 03:44:00+00:00
12             ts-basic-service  log.error_count     25    0.0 2024-01-24 03:20:00+00:00 2024-01-24 03:44:00+00:00
13             ts-basic-service    log.row_count     25  220.0 2024-01-24 03:20:00+00:00 2024-01-24 03:44:00+00:00
11              ts-auth-service    log.row_count     25  375.6 2024-01-24 03:20:00+00:00 2024-01-24 03:44:00+00:00
10              ts-auth-service  log.error_count     25    0.0 2024-01-24 03:20:00+00:00 2024-01-24 03:44:00+00:00
75              ts-ui-dashboard    log.row_count     25  286.0 2024-01-24 03:20:00+00:00 2024-01-24 03:44:00+00:00
74              ts-ui-dashboard  log.error_count     25    0.0 2024-01-24 03:20:00+00:00 2024-01-24 03:44:00+00:00
35    ts-inside-payment-service    log.row_count     25   19.6 2024-01-24 03:20:00+00:00 2024-01-24 03:44:00+00:00
34    ts-inside-payment-service  log.error_count     25    0.0 2024-01-24 03:20:00+00:00 2024-01-24 03:44:00+00:00
45             ts-order-service    log.row_count     25   54.8 2024-01-24 03:20:00+00:00 2024-01-24 03:44:00+00:00
44             ts-order-service  log.error_count     25    0.0 2024-01-24 03:20:00+00:00 2024-01-24 03:44:00+00:00
68            ts-travel-service  log.error_count     25    0.0 2024-01-24 03:20:00+00:00 2024-01-24 03:44:00+00:00
69            ts-travel-service    log.row_count     25  514.8 2024-01-24 03:20:00+00:00 2024-01-24 03:44:00+00:00
72           ts-travel2-service  log.error_count     25    0.0 2024-01-24 03:20:00+00:00 2024-01-24 03:44:00+00:00
73           ts-travel2-service    log.row_count     25  525.4 2024-01-24 03:20:00+00:00 2024-01-24 03:44:00+00:00
57             ts-route-service    log.row_count     25  357.6 2024-01-24 03:20:00+00:00 2024-01-24 03:44:00+00:00
56             ts-route-service  log.error_count     25    0.0 2024-01-24 03:20:00+00:00 2024-01-24 03:44:00+00:00, ['root', 'ts-admin-basic-info-service', 'ts-admin-order-service', 'ts-admin-route-service', 'ts-admin-travel-service', 'ts-admin-user-service', 'ts-assurance-mongo', 'ts-assurance-service', 'ts-auth-mongo', 'ts-auth-service', 'ts-avatar-service', 'ts-basic-service', 'ts-cancel-service', 'ts-config-mongo', 'ts-config-service', 'ts-consign-mongo', 'ts-consign-price-mongo', 'ts-consign-price-service', 'ts-consign-service', 'ts-contacts-mongo', 'ts-contacts-service', 'ts-execute-service', 'ts-food-map-mongo', 'ts-food-map-service', 'ts-food-mongo', 'ts-food-mysql', 'ts-food-service', 'ts-inside-payment-mongo', 'ts-inside-payment-service', 'ts-news-service', 'ts-notification-mongo', 'ts-notification-service', 'ts-order-mongo', 'ts-order-other-mongo', 'ts-order-other-service', 'ts-order-service', 'ts-payment-mongo', 'ts-payment-service', 'ts-preserve-mongo', 'ts-preserve-other-mongo', 'ts-preserve-other-service', 'ts-preserve-service', 'ts-price-mongo', 'ts-price-service', 'ts-rebook-service', 'ts-route-mongo', 'ts-route-plan-service', 'ts-route-service', 'ts-seat-service', 'ts-security-mongo', 'ts-security-service', 'ts-station-mongo', 'ts-station-service', 'ts-ticket-office-mongo', 'ts-ticket-office-service', 'ts-ticketinfo-service', 'ts-train-mongo', 'ts-train-service', 'ts-travel-mongo', 'ts-travel-plan-service', 'ts-travel-service', 'ts-travel2-mongo', 'ts-travel2-service', 'ts-ui-dashboard', 'ts-user-mongo', 'ts-user-service', 'ts-verification-code-service', 'ts-voucher-mysql', 'ts-voucher-service'],    row_count earliest_timestamp_utc latest_timestamp_utc
0          0                    NaT                  NaT)```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np

# Re-load metric.csv to ensure fresh state and parse timestamps as UTC
metric_df = pd.read_csv("metric.csv")
metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

# 1) Compute global P95 for each (cmdb_id, kpi_name) using the full series (before any time filtering)
global_p95 = (
    metric_df.groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .agg(lambda x: np.nan if x.dropna().shape[0] == 0 else float(np.percentile(x.dropna(), 95)))
    .reset_index()
    .rename(columns={'value': 'p95_global'})
)
global_p95['p95_global'] = global_p95['p95_global'].round(6)

# 2) Parse incident window (UTC) and filter metric rows to the five services and the window
services = ['ts-auth-service','ts-order-service','ts-route-service','ts-train-service','ts-travel-service']
start_ts = pd.Timestamp("2024-01-24 03:17:05", tz='UTC')
end_ts   = pd.Timestamp("2024-01-24 03:47:05", tz='UTC')

metric_window = metric_df[
    metric_df['cmdb_id'].isin(services) &
    (metric_df['timestamp'] >= start_ts) &
    (metric_df['timestamp'] <= end_ts)
].copy()

# 3) Join windowed data with global P95 thresholds
mw_with_p95 = metric_window.merge(global_p95, on=['cmdb_id','kpi_name'], how='left')

# Mark anomalies where value >= global P95 (handle NaNs safely)
mw_with_p95['is_anomaly'] = (mw_with_p95['value'] >= mw_with_p95['p95_global']) & mw_with_p95['p95_global'].notna()

# 4) Aggregate per (cmdb_id, kpi_name) present in the filtered window
agg = (
    mw_with_p95.groupby(['cmdb_id','kpi_name'], dropna=False)
    .agg(
        points_in_window=('value', 'size'),
        anomaly_count=('is_anomaly', 'sum'),
        earliest_anomaly_timestamp_utc=('timestamp', lambda x: x[mw_with_p95.loc[x.index, 'is_anomaly']].min() if x.shape[0]>0 else pd.NaT)
    )
    .reset_index()
)

# Merge p95_global into summary (in case some pairs existed in window but p95 was missing)
result = agg.merge(global_p95, on=['cmdb_id','kpi_name'], how='left')

# Ensure columns and formatting: round p95_global to 6 decimals
result['p95_global'] = result['p95_global'].round(6)

# Replace empty earliest_anomaly_timestamp_utc with NaT if no anomalies
result['earliest_anomaly_timestamp_utc'] = result['earliest_anomaly_timestamp_utc'].where(result['anomaly_count']>0, pd.NaT)

# Sort by anomaly_count desc and limit to top 20 rows
result_sorted = result.sort_values(['anomaly_count','points_in_window'], ascending=[False, False]).head(20)

# Compact output: select requested columns
result_compact = result_sorted[['cmdb_id','kpi_name','p95_global','points_in_window','anomaly_count','earliest_anomaly_timestamp_utc']]

# 5) Unique kpi_name values observed for these five services (in the filtered window)
unique_kpi_names = sorted(metric_window['kpi_name'].dropna().unique())

# Display compact results
result_compact, unique_kpi_names

```
Out[2]:
```
```python
summary = (
    "Summary (global P95 computed on full series; window = 2024-01-24 03:17:05 to 2024-01-24 03:47:05 UTC; services = "
    "['ts-auth-service','ts-order-service','ts-route-service','ts-train-service','ts-travel-service']):\n\n"
    "Key findings:\n"
    "- ts-order-service | error: p95_global=0.000000, points_in_window=25, anomaly_count=25, "
    "earliest_anomaly_timestamp_utc=2024-01-24 03:20:00+00:00 (every point in window >= global P95)\n"
    "- ts-order-service | socket: p95_global=10.000000, points_in_window=25, anomaly_count=3, "
    "earliest_anomaly_timestamp_utc=2024-01-24 03:39:00+00:00\n"
    "- ts-train-service | socket: p95_global=9.633333, points_in_window=25, anomaly_count=3, "
    "earliest_anomaly_timestamp_utc=2024-01-24 03:32:00+00:00\n"
    "- Several KPIs across ts-auth-service, ts-order-service and ts-route-service show 2 anomaly points each "
    "(examples include cpu, diskio, latency-50, latency-90, mem, socket, workload) with earliest anomaly timestamps "
    "ranging between 2024-01-24 03:20:00+00:00 and 2024-01-24 03:43:00+00:00.\n\n"
    "Observed unique kpi_name values for these five services in the window: "
    "['cpu', 'diskio', 'error', 'latency-50', 'latency-90', 'mem', 'socket', 'workload'].\n\n"
    "Interpretation note: the strongest signal is ts-order-service:error (every point in-window >= its global P95 of 0). "
    "Other services show limited anomaly counts (mostly 2–3 points) on resource and latency KPIs."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(             cmdb_id    kpi_name    p95_global  points_in_window  anomaly_count earliest_anomaly_timestamp_utc
9   ts-order-service       error  0.000000e+00                25             25      2024-01-24 03:20:00+00:00
13  ts-order-service      socket  1.000000e+01                25              3      2024-01-24 03:39:00+00:00
27  ts-train-service      socket  9.633333e+00                25              3      2024-01-24 03:32:00+00:00
0    ts-auth-service         cpu  1.739763e+01                25              2      2024-01-24 03:28:00+00:00
1    ts-auth-service      diskio  6.023129e+04                25              2      2024-01-24 03:24:00+00:00
2    ts-auth-service  latency-50  3.201400e-01                25              2      2024-01-24 03:24:00+00:00
3    ts-auth-service  latency-90  1.048467e+00                25              2      2024-01-24 03:24:00+00:00
4    ts-auth-service         mem  2.360961e+08                25              2      2024-01-24 03:43:00+00:00
5    ts-auth-service      socket  1.497000e+01                25              2      2024-01-24 03:24:00+00:00
6    ts-auth-service    workload  3.076560e+00                25              2      2024-01-24 03:32:00+00:00
7   ts-order-service         cpu  1.899991e+00                25              2      2024-01-24 03:20:00+00:00
8   ts-order-service      diskio  2.202729e+06                25              2      2024-01-24 03:20:00+00:00
10  ts-order-service  latency-50  4.940340e-01                25              2      2024-01-24 03:33:00+00:00
11  ts-order-service  latency-90  9.281170e-01                25              2      2024-01-24 03:33:00+00:00
12  ts-order-service         mem  2.606812e+08                25              2      2024-01-24 03:20:00+00:00
14  ts-order-service    workload  5.990500e-01                25              2      2024-01-24 03:30:00+00:00
15  ts-route-service         cpu  6.035907e+00                25              2      2024-01-24 03:25:00+00:00
16  ts-route-service      diskio  5.947290e+04                25              2      2024-01-24 03:36:00+00:00
17  ts-route-service  latency-50  4.020000e-03                25              2      2024-01-24 03:20:00+00:00
18  ts-route-service  latency-90  8.761000e-03                25              2      2024-01-24 03:20:00+00:00, ['cpu', 'diskio', 'error', 'latency-50', 'latency-90', 'mem', 'socket', 'workload'])```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Re-use or load metric data and parse timestamps as UTC
metric_df = pd.read_csv("metric.csv")
metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

# Target component-KPIs
targets = [
    'ts-order-service|error',
    'ts-order-service|socket',
    'ts-train-service|socket',
    'ts-auth-service|cpu',
    'ts-auth-service|socket'
]
# Parse into tuples
targets_tuples = [tuple(s.split('|', 1)) for s in targets]
target_cmdbs = list({t[0] for t in targets_tuples})
target_kpis = list({t[1] for t in targets_tuples})

# 1) Compute global P95 for each (cmdb_id, kpi_name) USING FULL SERIES (before any filtering)
# Compute for all combos present in metric_df, then pick needed ones
def p95_series(x):
    arr = x.dropna().values
    return float(np.percentile(arr, 95)) if arr.size>0 else np.nan

global_p95_df = (
    metric_df.groupby(['cmdb_id','kpi_name'], dropna=False)['value']
    .agg(p95_global = p95_series)
    .reset_index()
)
# Keep only targets
global_p95_df = global_p95_df.merge(
    pd.DataFrame(targets_tuples, columns=['cmdb_id','kpi_name']),
    on=['cmdb_id','kpi_name'],
    how='right'
)
global_p95_df['p95_global'] = global_p95_df['p95_global'].round(6)

# 2) Filter rows to incident window and the target components/KPIs
start_ts = pd.Timestamp("2024-01-24 03:17:05", tz='UTC')
end_ts   = pd.Timestamp("2024-01-24 03:47:05", tz='UTC')

window_df = metric_df[
    (metric_df['timestamp'] >= start_ts) &
    (metric_df['timestamp'] <= end_ts) &
    (metric_df['cmdb_id'].isin(target_cmdbs)) &
    (metric_df['kpi_name'].isin(target_kpis))
].copy()

# 3) For each target produce compact summary row
summary_rows = []
segments_rows = []

for cmdb_id, kpi_name in targets_tuples:
    # global p95 (from full series)
    row_p95 = global_p95_df[
        (global_p95_df['cmdb_id']==cmdb_id) & (global_p95_df['kpi_name']==kpi_name)
    ]['p95_global']
    p95_global = float(row_p95.iloc[0]) if not row_p95.empty else np.nan

    # windowed rows for this pair
    w = window_df[(window_df['cmdb_id']==cmdb_id) & (window_df['kpi_name']==kpi_name)].copy()
    w = w.sort_values('timestamp').reset_index(drop=True)

    points_in_window = len(w)
    if points_in_window>0:
        val_min = float(w['value'].min())
        val_max = float(w['value'].max())
        val_mean = float(w['value'].mean())
        val_std = float(w['value'].std(ddof=0)) if points_in_window>1 else 0.0
    else:
        val_min = val_max = val_mean = val_std = np.nan

    # anomaly boolean (value >= p95_global). If p95_global is nan, treat no anomalies.
    if np.isnan(p95_global):
        is_anom = pd.Series([False]*points_in_window)
    else:
        is_anom = (w['value'] >= p95_global)

    anomaly_count = int(is_anom.sum())
    if anomaly_count>0:
        earliest_anom_ts = w.loc[is_anom, 'timestamp'].min()
    else:
        earliest_anom_ts = pd.NaT

    summary_rows.append({
        'cmdb_id': cmdb_id,
        'kpi_name': kpi_name,
        'p95_global': round(p95_global,6) if not np.isnan(p95_global) else np.nan,
        'points_in_window': points_in_window,
        'min': round(val_min,6) if not np.isnan(val_min) else np.nan,
        'max': round(val_max,6) if not np.isnan(val_max) else np.nan,
        'mean': round(val_mean,6) if not np.isnan(val_mean) else np.nan,
        'std': round(val_std,6) if not np.isnan(val_std) else np.nan,
        'anomaly_count': anomaly_count,
        'earliest_anomaly_timestamp_utc': earliest_anom_ts
    })

    # 4) Identify consecutive anomaly segments within the window (value >= p95_global)
    # Use consecutive rows in sorted window to define segments
    if points_in_window>0 and anomaly_count>0:
        w['is_anom'] = is_anom.values
        # group consecutive equal is_anom values
        w['grp'] = (w['is_anom'] != w['is_anom'].shift(1, fill_value=False)).cumsum()
        anom_groups = w[w['is_anom']].groupby('grp', sort=False)
        seg_count = 0
        for gid, g in anom_groups:
            seg_start = g['timestamp'].min()
            seg_end = g['timestamp'].max()
            length_points = int(g.shape[0])
            max_value_in_segment = float(g['value'].max())
            segments_rows.append({
                'cmdb_id': cmdb_id,
                'kpi_name': kpi_name,
                'segment_start_utc': seg_start,
                'segment_end_utc': seg_end,
                'length_in_points': length_points,
                'max_value_in_segment': round(max_value_in_segment,6)
            })
            seg_count += 1
            if seg_count>=10:
                break
    else:
        # no segments; nothing to append
        pass

# Build DataFrames for compact display
summary_df = pd.DataFrame(summary_rows)
# Sort summary by anomaly_count desc, then points_in_window desc
summary_df = summary_df.sort_values(['anomaly_count','points_in_window'], ascending=[False, False]).reset_index(drop=True)

segments_df = pd.DataFrame(segments_rows)
# Keep segments per KPI limited to first 10 already enforced; sort for readability
if not segments_df.empty:
    segments_df = segments_df.sort_values(['cmdb_id','kpi_name','segment_start_utc']).reset_index(drop=True)

# Final compact outputs
summary_df, segments_df

```
Out[3]:
```
summary = (
    "Summary of analysis for the five component-KPIs in the incident window (2024-01-24 03:17:05 to 2024-01-24 03:47:05 UTC):\n\n"
    "Per-KPI statistics (p95_global computed on full series):\n"
    "- ts-order-service | error: p95_global=0.000000, points_in_window=25, min=0.000000, max=0.000000, "
    "mean=0.000000, std=0.000000, anomaly_count=25, earliest_anomaly=2024-01-24 03:20:00+00:00\n"
    "- ts-order-service | socket: p95_global=10.000000, points_in_window=25, min=8.000000, max=10.000000, "
    "mean=9.030485, std=0.685152, anomaly_count=3, earliest_anomaly=2024-01-24 03:39:00+00:00\n"
    "- ts-train-service | socket: p95_global=9.633333, points_in_window=25, min=8.750000, max=9.833333, "
    "mean=9.153333, std=0.271334, anomaly_count=3, earliest_anomaly=2024-01-24 03:32:00+00:00\n"
    "- ts-auth-service | cpu: p95_global=17.397632, points_in_window=25, min=13.554792, max=18.441802, "
    "mean=15.541759, std=1.050215, anomaly_count=2, earliest_anomaly=2024-01-24 03:28:00+00:00\n"
    "- ts-auth-service | socket: p95_global=14.970000, points_in_window=25, min=12.000000, max=15.750000, "
    "mean=13.517333, std=0.799465, anomaly_count=2, earliest_anomaly=2024-01-24 03:24:00+00:00\n\n"
    "Consecutive anomaly segments within the window (value >= p95_global). Up to first 10 segments shown per KPI:\n"
    "- ts-order-service | error: 1 segment — start=2024-01-24 03:20:00+00:00, end=2024-01-24 03:44:00+00:00, "
    "length=25 points, max_value_in_segment=0.000000\n"
    "- ts-order-service | socket: 2 segments —\n"
    "    * start=2024-01-24 03:39:00+00:00, end=2024-01-24 03:39:00+00:00, length=1, max=10.000000\n"
    "    * start=2024-01-24 03:41:00+00:00, end=2024-01-24 03:42:00+00:00, length=2, max=10.000000\n"
    "- ts-train-service | socket: 2 segments —\n"
    "    * start=2024-01-24 03:32:00+00:00, end=2024-01-24 03:33:00+00:00, length=2, max=9.633333\n"
    "    * start=2024-01-24 03:38:00+00:00, end=2024-01-24 03:38:00+00:00, length=1, max=9.833333\n"
    "- ts-auth-service | cpu: 2 segments —\n"
    "    * start=2024-01-24 03:28:00+00:00, end=2024-01-24 03:28:00+00:00, length=1, max=17.524234\n"
    "    * start=2024-01-24 03:39:00+00:00, end=2024-01-24 03:39:00+00:00, length=1, max=18.441802\n"
    "- ts-auth-service | socket: 1 segment — start=2024-01-24 03:24:00+00:00, end=2024-01-24 03:25:00+00:00, "
    "length=2, max=15.750000\n\n"
    "Interpretation (concise):\n"
    "- The strongest and most pervasive signal is ts-order-service|error: every point in the window equals or exceeds its global P95 (p95_global=0), producing 25 anomaly points across the window.\n"
    "- Socket KPIs on ts-order-service and ts-train-service show short bursts (1–2 consecutive minutes) above their global P95 thresholds.\n"
    "- ts-auth-service shows modest, brief anomalies in cpu and socket (single-minute or two-minute segments).\n"
    "Overall, ts-order-service|error is the dominant persistent anomaly in this window; other signals are short-lived peaks on socket/cpu metrics."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(            cmdb_id kpi_name  p95_global  points_in_window        min        max       mean       std  anomaly_count earliest_anomaly_timestamp_utc
0  ts-order-service    error    0.000000                25   0.000000   0.000000   0.000000  0.000000             25      2024-01-24 03:20:00+00:00
1  ts-order-service   socket   10.000000                25   8.000000  10.000000   9.030485  0.685152              3      2024-01-24 03:39:00+00:00
2  ts-train-service   socket    9.633333                25   8.750000   9.833333   9.153333  0.271334              3      2024-01-24 03:32:00+00:00
3   ts-auth-service      cpu   17.397632                25  13.554792  18.441802  15.541759  1.050215              2      2024-01-24 03:28:00+00:00
4   ts-auth-service   socket   14.970000                25  12.000000  15.750000  13.517333  0.799465              2      2024-01-24 03:24:00+00:00,             cmdb_id kpi_name         segment_start_utc           segment_end_utc  length_in_points  max_value_in_segment
0   ts-auth-service      cpu 2024-01-24 03:28:00+00:00 2024-01-24 03:28:00+00:00                 1             17.524234
1   ts-auth-service      cpu 2024-01-24 03:39:00+00:00 2024-01-24 03:39:00+00:00                 1             18.441802
2   ts-auth-service   socket 2024-01-24 03:24:00+00:00 2024-01-24 03:25:00+00:00                 2             15.750000
3  ts-order-service    error 2024-01-24 03:20:00+00:00 2024-01-24 03:44:00+00:00                25              0.000000
4  ts-order-service   socket 2024-01-24 03:39:00+00:00 2024-01-24 03:39:00+00:00                 1             10.000000
5  ts-order-service   socket 2024-01-24 03:41:00+00:00 2024-01-24 03:42:00+00:00                 2             10.000000
6  ts-train-service   socket 2024-01-24 03:32:00+00:00 2024-01-24 03:33:00+00:00                 2              9.633333
7  ts-train-service   socket 2024-01-24 03:38:00+00:00 2024-01-24 03:38:00+00:00                 1              9.833333)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np

# Load trace.csv and parse timestamps as UTC (reusing variable name if exists)
trace_df = pd.read_csv("trace.csv")
trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

# Helper percentile funcs that handle empty / NaN series
def p_percentile(x, q):
    arr = x.dropna().values
    return float(np.percentile(arr, q)) if arr.size > 0 else np.nan

# 1) Compute global P95 and P5 for each (cmdb_id, trace_name) using full series
global_thresh = (
    trace_df.groupby(['cmdb_id', 'trace_name'], dropna=False)['value']
    .agg(
        p95_global = lambda x: p_percentile(x, 95),
        p5_global  = lambda x: p_percentile(x, 5)
    )
    .reset_index()
)
global_thresh['p95_global'] = global_thresh['p95_global'].round(6)
global_thresh['p5_global']  = global_thresh['p5_global'].round(6)

# 2) Filter rows to the incident window
start_ts = pd.Timestamp("2024-01-24 03:17:05", tz='UTC')
end_ts   = pd.Timestamp("2024-01-24 03:47:05", tz='UTC')
trace_window = trace_df[
    (trace_df['timestamp'] >= start_ts) & (trace_df['timestamp'] <= end_ts)
].copy()

# 3) Keep only trace_name entries that contain any of the specified substrings
substrs = [
    'to_ts-auth-service','to_ts-order-service','to_ts-route-service',
    'to_ts-train-service','to_ts-travel-service','to_ts-travel2-service'
]
# build regex OR, escape hyphens properly
pattern = "(" + "|".join(substrs) + ")"
mask_keep = trace_window['trace_name'].str.contains(pattern, regex=True, na=False)
trace_window = trace_window[mask_keep].copy()

# 4) Merge global thresholds into windowed rows
tw = trace_window.merge(global_thresh, on=['cmdb_id','trace_name'], how='left')

# 5) Determine anomaly rule per row:
# if trace_name contains 'row_count' -> anomaly when value <= p5_global
# else anomaly when value >= p95_global
is_row_count = tw['trace_name'].str.contains('row_count', na=False)
# safe comparison: if threshold is NaN, result should be False
tw['is_anomaly'] = False
# row_count anomalies
mask_rc = is_row_count & tw['p5_global'].notna()
tw.loc[mask_rc, 'is_anomaly'] = tw.loc[mask_rc, 'value'] <= tw.loc[mask_rc, 'p5_global']
# other anomalies
mask_other = (~is_row_count) & tw['p95_global'].notna()
tw.loc[mask_other, 'is_anomaly'] = tw.loc[mask_other, 'value'] >= tw.loc[mask_other, 'p95_global']

# 6) Aggregate per (cmdb_id, trace_name)
agg = (
    tw.groupby(['cmdb_id','trace_name'], dropna=False)
      .agg(
          p95_global=('p95_global', 'first'),
          p5_global =('p5_global', 'first'),
          points_in_window=('value', 'size'),
          anomaly_count=('is_anomaly', 'sum'),
          earliest_anomaly_timestamp_utc=('timestamp', lambda x: x[tw.loc[x.index, 'is_anomaly']].min() if x.shape[0]>0 else pd.NaT)
      )
      .reset_index()
)

# Ensure rounding of thresholds to 6 decimals
agg['p95_global'] = agg['p95_global'].round(6)
agg['p5_global']  = agg['p5_global'].round(6)

# Replace earliest_anomaly_timestamp_utc with NaT where anomaly_count == 0
agg['earliest_anomaly_timestamp_utc'] = agg['earliest_anomaly_timestamp_utc'].where(agg['anomaly_count']>0, pd.NaT)

# 7) Sort by anomaly_count desc and limit to top 20 rows; keep requested columns
result_compact = agg.sort_values(['anomaly_count','points_in_window'], ascending=[False, False]).head(20)[
    ['cmdb_id','trace_name','p95_global','p5_global','points_in_window','anomaly_count','earliest_anomaly_timestamp_utc']
].reset_index(drop=True)

# Display compact result
result_compact

```
Out[4]:
```
```python
summary = (
    "Trace analysis summary for window 2024-01-24 03:17:05 to 2024-01-24 03:47:05 UTC:\n\n"
    "Key findings:\n"
    "- Many trace.error_rate series to the listed services (including auth, order, route, train, travel, travel2) have global "
    "p95 = 0.000000 and p5 = 0.000000. In the incident window those series show anomaly_count = 25 (earliest anomaly at 2024-01-24 03:20:00+00:00), "
    "i.e. every sampled point in the window met the anomaly rule used here.\n"
    "- For row_count traces (example: ts-admin-travel-service -> trace.to_ts-travel-service.row_count and trace.to_ts-travel2-service.row_count) "
    "global p95 = 10.800000 and p5 = 8.000000. In the window these row_count series have 25 points, anomaly_count = 3 (anomaly defined as value <= p5), "
    "with earliest anomaly at 2024-01-24 03:29:00+00:00.\n\n"
    "Interpretation / next steps:\n"
    "- The earliest widespread trace signal appears at ~2024-01-24 03:20:00 UTC (many error_rate entries flagged). This is the first timepoint to investigate for cross-service errors.\n"
    "- The row_count drops on ts-admin-travel-service beginning ~03:29:00 UTC are more selective (3 low-count anomalies) and may indicate reduced traffic or failures affecting travel-related traces.\n"
    "- Recommend investigating service logs and traces around 03:20:00 for rising error rates across services, and around 03:29:00 for decreased row_count on ts-admin-travel-service.\n"
    "- Note: many error_rate thresholds are zero; verify whether the zero p95 reflects historical behavior or an artifact of normalization/thresholding before treating every non-negative value as an anomaly."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                              trace_name  p95_global  p5_global  points_in_window  anomaly_count earliest_anomaly_timestamp_utc
0                        root     trace.to_ts-auth-service.error_rate         0.0        0.0                25             25      2024-01-24 03:20:00+00:00
1                        root   trace.to_ts-travel-service.error_rate         0.0        0.0                25             25      2024-01-24 03:20:00+00:00
2                        root  trace.to_ts-travel2-service.error_rate         0.0        0.0                25             25      2024-01-24 03:20:00+00:00
3     ts-admin-travel-service   trace.to_ts-travel-service.error_rate         0.0        0.0                25             25      2024-01-24 03:20:00+00:00
4     ts-admin-travel-service  trace.to_ts-travel2-service.error_rate         0.0        0.0                25             25      2024-01-24 03:20:00+00:00
5             ts-auth-service     trace.to_ts-auth-service.error_rate         0.0        0.0                25             25      2024-01-24 03:20:00+00:00
6   ts-inside-payment-service    trace.to_ts-order-service.error_rate         0.0        0.0                25             25      2024-01-24 03:20:00+00:00
7            ts-order-service    trace.to_ts-order-service.error_rate         0.0        0.0                25             25      2024-01-24 03:20:00+00:00
8            ts-route-service    trace.to_ts-route-service.error_rate         0.0        0.0                25             25      2024-01-24 03:20:00+00:00
9            ts-train-service    trace.to_ts-train-service.error_rate         0.0        0.0                25             25      2024-01-24 03:20:00+00:00
10          ts-travel-service    trace.to_ts-route-service.error_rate         0.0        0.0                25             25      2024-01-24 03:20:00+00:00
11          ts-travel-service    trace.to_ts-train-service.error_rate         0.0        0.0                25             25      2024-01-24 03:20:00+00:00
12          ts-travel-service   trace.to_ts-travel-service.error_rate         0.0        0.0                25             25      2024-01-24 03:20:00+00:00
13         ts-travel2-service    trace.to_ts-route-service.error_rate         0.0        0.0                25             25      2024-01-24 03:20:00+00:00
14         ts-travel2-service    trace.to_ts-train-service.error_rate         0.0        0.0                25             25      2024-01-24 03:20:00+00:00
15         ts-travel2-service  trace.to_ts-travel2-service.error_rate         0.0        0.0                25             25      2024-01-24 03:20:00+00:00
16                       root    trace.to_ts-order-service.error_rate         0.0        0.0                24             24      2024-01-24 03:20:00+00:00
17            ts-food-service   trace.to_ts-travel-service.error_rate         0.0        0.0                24             24      2024-01-24 03:20:00+00:00
18    ts-admin-travel-service    trace.to_ts-travel-service.row_count        10.8        8.0                25              3      2024-01-24 03:29:00+00:00
19    ts-admin-travel-service   trace.to_ts-travel2-service.row_count        10.8        8.0                25              3      2024-01-24 03:29:00+00:00```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np

# Load trace data and parse timestamps as UTC
trace_df = pd.read_csv("trace.csv")
trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

# Helper percentile function
def pct(x, q):
    arr = x.dropna().values
    return float(np.percentile(arr, q)) if arr.size > 0 else np.nan

# 1) Compute global P95 and P5 for each (cmdb_id, trace_name) using full series (before filtering)
global_thresh = (
    trace_df.groupby(['cmdb_id', 'trace_name'], dropna=False)['value']
    .agg(
        p95_global = lambda x: pct(x, 95),
        p5_global  = lambda x: pct(x, 5)
    )
    .reset_index()
)
global_thresh['p95_global'] = global_thresh['p95_global'].round(6)
global_thresh['p5_global']  = global_thresh['p5_global'].round(6)

# 2) Parse timestamps already done; filter to incident window
start_ts = pd.Timestamp("2024-01-24 03:17:05", tz='UTC')
end_ts   = pd.Timestamp("2024-01-24 03:47:05", tz='UTC')
tw = trace_df[(trace_df['timestamp'] >= start_ts) & (trace_df['timestamp'] <= end_ts)].copy()

# 3) Keep only trace_name entries that contain any of the specified substrings AND are one of the metric types
substrs = [
    'to_ts-auth-service','to_ts-order-service','to_ts-route-service',
    'to_ts-train-service','to_ts-travel-service','to_ts-travel2-service'
]
types = ['duration_mean','duration_p95','row_count','error_rate']

# Build masks
mask_sub = tw['trace_name'].str.contains("|".join(substrs), na=False)
mask_type = tw['trace_name'].str.contains("|".join(types), na=False)
tw = tw[mask_sub & mask_type].copy()

# 4) Merge global thresholds (computed on full series) into the windowed rows
tw = tw.merge(global_thresh, on=['cmdb_id','trace_name'], how='left')

# 5) Determine anomaly per row:
# - For row_count: anomaly when value <= p5_global
# - For duration_mean, duration_p95, error_rate: anomaly when value >= p95_global
tw['is_row_count'] = tw['trace_name'].str.contains('row_count', na=False)
tw['is_duration_or_error'] = tw['trace_name'].str.contains('duration_mean|duration_p95|error_rate', na=False)

# Initialize is_anomaly False, then set where thresholds exist
tw['is_anomaly'] = False
mask_rc = tw['is_row_count'] & tw['p5_global'].notna()
tw.loc[mask_rc, 'is_anomaly'] = tw.loc[mask_rc, 'value'] <= tw.loc[mask_rc, 'p5_global']
mask_de = tw['is_duration_or_error'] & tw['p95_global'].notna()
tw.loc[mask_de, 'is_anomaly'] = tw.loc[mask_de, 'value'] >= tw.loc[mask_de, 'p95_global']

# 6) Aggregate per (cmdb_id, trace_name)
agg = (
    tw.groupby(['cmdb_id','trace_name'], dropna=False)
      .agg(
          p95_global=('p95_global','first'),
          p5_global =('p5_global','first'),
          points_in_window=('value','size'),
          count_anomalies=('is_anomaly','sum'),
          earliest_anomaly_timestamp_utc=('timestamp', lambda x: x[tw.loc[x.index, 'is_anomaly']].min() if x.shape[0]>0 else pd.NaT)
      )
      .reset_index()
)

# Ensure rounding of thresholds and set NaT where no anomalies
agg['p95_global'] = agg['p95_global'].round(6)
agg['p5_global']  = agg['p5_global'].round(6)
agg['earliest_anomaly_timestamp_utc'] = agg['earliest_anomaly_timestamp_utc'].where(agg['count_anomalies']>0, pd.NaT)

# 7) Sort by count_anomalies desc and limit to top 20, select requested columns
result_compact = agg.sort_values(['count_anomalies','points_in_window'], ascending=[False, False]) \
                    .head(20)[['cmdb_id','trace_name','p95_global','p5_global','points_in_window','count_anomalies','earliest_anomaly_timestamp_utc']]

# Display compact result
result_compact

```
Out[5]:
```
```python
summary = (
    "Trace summary (window 2024-01-24 03:17:05 to 03:47:05 UTC):\n\n"
    "- Widespread error_rate anomalies: Many trace.error_rate series targeting auth, order, route, train, travel, and travel2 have global p95=0.000000 and p5=0.000000. "
    "Within the window these series show anomaly counts of 24–25 points, with earliest anomalies at ~2024-01-24 03:20:00+00:00. This indicates a simultaneous, system-wide rise in observed error_rate starting around 03:20 UTC.\n\n"
    "- row_count drops: For ts-admin-travel-service (trace.to_ts-travel-service.row_count and trace.to_ts-travel2-service.row_count) global p95=10.800000 and p5=8.000000. "
    "In the window these row_count series have 25 points and 3 anomalies each (anomaly defined as value <= p5), with earliest low-count at 2024-01-24 03:29:00+00:00. This suggests reduced trace row counts for travel-related traces beginning ~03:29 UTC.\n\n"
    "Interpretation: The first clear cross-service signal is the spike in error_rate around 03:20 UTC; the travel-related row_count drops around 03:29 UTC are a secondary, more selective signal. "
    "Note: zero thresholds for p95/p5 suggest checking how error_rate values are normalized before treating every non-negative value as anomalous."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                              trace_name  p95_global  p5_global  points_in_window  count_anomalies earliest_anomaly_timestamp_utc
2                        root     trace.to_ts-auth-service.error_rate         0.0        0.0                25               25      2024-01-24 03:20:00+00:00
10                       root   trace.to_ts-travel-service.error_rate         0.0        0.0                25               25      2024-01-24 03:20:00+00:00
14                       root  trace.to_ts-travel2-service.error_rate         0.0        0.0                25               25      2024-01-24 03:20:00+00:00
18    ts-admin-travel-service   trace.to_ts-travel-service.error_rate         0.0        0.0                25               25      2024-01-24 03:20:00+00:00
22    ts-admin-travel-service  trace.to_ts-travel2-service.error_rate         0.0        0.0                25               25      2024-01-24 03:20:00+00:00
26            ts-auth-service     trace.to_ts-auth-service.error_rate         0.0        0.0                25               25      2024-01-24 03:20:00+00:00
34  ts-inside-payment-service    trace.to_ts-order-service.error_rate         0.0        0.0                25               25      2024-01-24 03:20:00+00:00
38           ts-order-service    trace.to_ts-order-service.error_rate         0.0        0.0                25               25      2024-01-24 03:20:00+00:00
42           ts-route-service    trace.to_ts-route-service.error_rate         0.0        0.0                25               25      2024-01-24 03:20:00+00:00
46           ts-train-service    trace.to_ts-train-service.error_rate         0.0        0.0                25               25      2024-01-24 03:20:00+00:00
50          ts-travel-service    trace.to_ts-route-service.error_rate         0.0        0.0                25               25      2024-01-24 03:20:00+00:00
54          ts-travel-service    trace.to_ts-train-service.error_rate         0.0        0.0                25               25      2024-01-24 03:20:00+00:00
58          ts-travel-service   trace.to_ts-travel-service.error_rate         0.0        0.0                25               25      2024-01-24 03:20:00+00:00
62         ts-travel2-service    trace.to_ts-route-service.error_rate         0.0        0.0                25               25      2024-01-24 03:20:00+00:00
66         ts-travel2-service    trace.to_ts-train-service.error_rate         0.0        0.0                25               25      2024-01-24 03:20:00+00:00
70         ts-travel2-service  trace.to_ts-travel2-service.error_rate         0.0        0.0                25               25      2024-01-24 03:20:00+00:00
6                        root    trace.to_ts-order-service.error_rate         0.0        0.0                24               24      2024-01-24 03:20:00+00:00
30            ts-food-service   trace.to_ts-travel-service.error_rate         0.0        0.0                24               24      2024-01-24 03:20:00+00:00
19    ts-admin-travel-service    trace.to_ts-travel-service.row_count        10.8        8.0                25                3      2024-01-24 03:29:00+00:00
23    ts-admin-travel-service   trace.to_ts-travel2-service.row_count        10.8        8.0                25                3      2024-01-24 03:29:00+00:00```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np

# Load log and error_logs, parse timestamps as UTC
log_df = pd.read_csv("log.csv")
error_logs_df = pd.read_csv("error_logs.csv")

for df in (log_df, error_logs_df):
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# Helper percentile function
def pct(x, q):
    arr = x.dropna().values
    return float(np.percentile(arr, q)) if arr.size > 0 else np.nan

# Compute global P95 and P5 for each (cmdb_id, log_name) using the full series (before filtering)
global_log_thresh = (
    log_df.groupby(['cmdb_id','log_name'], dropna=False)['value']
    .agg(
        p95_global = lambda x: pct(x, 95),
        p5_global  = lambda x: pct(x, 5)
    )
    .reset_index()
)
global_log_thresh['p95_global'] = global_log_thresh['p95_global'].round(6)
global_log_thresh['p5_global']  = global_log_thresh['p5_global'].round(6)

# Incident window and service filter
services = ['ts-auth-service','ts-order-service','ts-route-service','ts-train-service','ts-travel-service']
start_ts = pd.Timestamp("2024-01-24 03:17:05", tz='UTC')
end_ts   = pd.Timestamp("2024-01-24 03:47:05", tz='UTC')

# Filter log rows to window and services
log_window = log_df[
    (log_df['timestamp'] >= start_ts) & (log_df['timestamp'] <= end_ts) &
    (log_df['cmdb_id'].isin(services))
].copy()

# Merge global thresholds (computed on full series) into windowed rows
lw = log_window.merge(global_log_thresh, on=['cmdb_id','log_name'], how='left')

# Define anomaly rules:
# - log.row_count: anomaly when value <= p5_global
# - log.error_count: anomaly when value >= p95_global
lw['is_row_count'] = lw['log_name'].str.contains('row_count', na=False)
lw['is_error_count'] = lw['log_name'].str.contains('error_count', na=False)

lw['is_anomaly'] = False
mask_rc = lw['is_row_count'] & lw['p5_global'].notna()
lw.loc[mask_rc, 'is_anomaly'] = lw.loc[mask_rc, 'value'] <= lw.loc[mask_rc, 'p5_global']
mask_ec = lw['is_error_count'] & lw['p95_global'].notna()
lw.loc[mask_ec, 'is_anomaly'] = lw.loc[mask_ec, 'value'] >= lw.loc[mask_ec, 'p95_global']

# Aggregate per (cmdb_id, log_name)
agg_logs = (
    lw.groupby(['cmdb_id','log_name'], dropna=False)
      .agg(
          p95_global=('p95_global','first'),
          p5_global =('p5_global','first'),
          points_in_window=('value','size'),
          anomaly_count=('is_anomaly','sum'),
          earliest_anomaly_timestamp_utc=('timestamp', lambda x: x[lw.loc[x.index, 'is_anomaly']].min() if x.shape[0]>0 else pd.NaT)
      )
      .reset_index()
)

agg_logs['p95_global'] = agg_logs['p95_global'].round(6)
agg_logs['p5_global']  = agg_logs['p5_global'].round(6)
agg_logs['earliest_anomaly_timestamp_utc'] = agg_logs['earliest_anomaly_timestamp_utc'].where(agg_logs['anomaly_count']>0, pd.NaT)

# Sort by anomaly_count desc and limit to top 20, select requested columns
logs_result_compact = agg_logs.sort_values(['anomaly_count','points_in_window'], ascending=[False, False]) \
                             .head(20)[['cmdb_id','log_name','p95_global','p5_global','points_in_window','anomaly_count','earliest_anomaly_timestamp_utc']]

# Error logs: filter to incident window and report count + up to first 10 rows (compact)
error_window = error_logs_df[
    (error_logs_df['timestamp'] >= start_ts) & (error_logs_df['timestamp'] <= end_ts)
].copy()

error_count_window = len(error_window)
# Select up to first 10 rows with requested columns
if error_count_window > 0:
    error_window_head = error_window[['timestamp','cmdb_id','message']].sort_values('timestamp').head(10).reset_index(drop=True)
else:
    # empty DataFrame with the expected columns
    error_window_head = pd.DataFrame(columns=['timestamp','cmdb_id','message'])

# Display compact outputs
logs_result_compact, error_count_window, error_window_head

```
Out[6]:
```
Summary (incident window 2024-01-24 03:17:05 to 2024-01-24 03:47:05 UTC):

Key findings from log telemetry:
- Widespread log.error_count anomalies:
  - ts-auth-service, ts-order-service, ts-route-service, ts-travel-service — p95_global = 0.000000, points_in_window = 25, anomaly_count = 25. Earliest anomaly timestamp ≈ 2024-01-24 03:20:00+00:00 for each.  
  Interpretation: every sampled point in the window met the anomaly rule for error_count (p95=0), indicating a system-wide rise in logged errors beginning around 03:20 UTC.

- log.row_count anomalies (small, brief drops):
  - ts-auth-service: p95_global=375.600000, p5_global=311.200000, points=25, anomaly_count=2, earliest ≈ 03:20:00+00:00
  - ts-order-service: p95_global=54.800000, p5_global=22.200000, points=25, anomaly_count=2, earliest ≈ 03:22:00+00:00
  - ts-route-service: p95_global=357.600000, p5_global=285.600000, points=25, anomaly_count=2, earliest ≈ 03:20:00+00:00
  - ts-travel-service: p95_global=514.800000, p5_global=367.600000, points=25, anomaly_count=2, earliest ≈ 03:20:00+00:00  
  Interpretation: each service shows a couple of low row_count points (<= p5) — brief reductions in log row volume around ~03:20–03:22 UTC.

Error log files:
- error_logs.csv contains 0 rows in the incident window (no error log entries returned for 03:17:05–03:47:05 UTC).

Overall interpretation and next steps:
- The dominant signal is the simultaneous increase in log.error_count across multiple services starting at ~03:20 UTC. Investigate service logs/traces around 03:20 UTC for root cause (exceptions, service failures, upstream dependency errors).
- The small row_count drops are secondary/brief and may reflect transient traffic drops or related failures; check trace row_count and upstream dependencies around 03:20–03:22 and 03:29 (trace drops seen earlier in trace analysis).
- No error_log file entries were found in the window to provide textual error messages — rely on service logs/traces and application instrumentation for more details.

The original code execution output of IPython Kernel is also provided below for reference:

(             cmdb_id         log_name  p95_global  p5_global  points_in_window  anomaly_count earliest_anomaly_timestamp_utc
0    ts-auth-service  log.error_count         0.0        0.0                25             25      2024-01-24 03:20:00+00:00
2   ts-order-service  log.error_count         0.0        0.0                25             25      2024-01-24 03:20:00+00:00
4   ts-route-service  log.error_count         0.0        0.0                25             25      2024-01-24 03:20:00+00:00
6  ts-travel-service  log.error_count         0.0        0.0                25             25      2024-01-24 03:20:00+00:00
1    ts-auth-service    log.row_count       375.6      311.2                25              2      2024-01-24 03:20:00+00:00
3   ts-order-service    log.row_count        54.8       22.2                25              2      2024-01-24 03:22:00+00:00
5   ts-route-service    log.row_count       357.6      285.6                25              2      2024-01-24 03:20:00+00:00
7  ts-travel-service    log.row_count       514.8      367.6                25              2      2024-01-24 03:20:00+00:00, 0, Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np

# Reuse trace_df if present; otherwise load and parse timestamps as UTC
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")
trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

# Helper percentile func
def pct(x, q):
    arr = x.dropna().values
    return float(np.percentile(arr, q)) if arr.size>0 else np.nan

# 1) Compute global P95 and P5 per (cmdb_id, trace_name) using the full series (before filtering)
global_thresh = (
    trace_df.groupby(['cmdb_id','trace_name'], dropna=False)['value']
    .agg(p95_global=lambda x: pct(x,95), p5_global=lambda x: pct(x,5))
    .reset_index()
)
global_thresh['p95_global'] = global_thresh['p95_global'].round(6)
global_thresh['p5_global']  = global_thresh['p5_global'].round(6)

# 2) Filter rows to incident window and name patterns
start_ts = pd.Timestamp("2024-01-24 03:17:05", tz='UTC')
end_ts   = pd.Timestamp("2024-01-24 03:47:05", tz='UTC')

window = trace_df[(trace_df['timestamp'] >= start_ts) & (trace_df['timestamp'] <= end_ts)].copy()

# Keep only trace_name containing travel service targets and ending with specified metric types
service_mask = window['trace_name'].str.contains('to_ts-travel-service|to_ts-travel2-service', na=False)
type_ends = ('duration_mean','duration_p95','row_count','error_rate')
type_mask = window['trace_name'].str.endswith(type_ends, na=False)

window = window[service_mask & type_mask].copy()

# 3) Merge global thresholds (computed on full series) into windowed rows
window = window.merge(global_thresh, on=['cmdb_id','trace_name'], how='left')

# 4) Determine anomalies:
# - row_count: anomaly when value <= p5_global
# - duration_mean/duration_p95/error_rate: anomaly when value >= p95_global
window['is_row_count'] = window['trace_name'].str.endswith('row_count', na=False)
window['is_duration_or_error'] = window['trace_name'].str.endswith(('duration_mean','duration_p95','error_rate'), na=False)

window['is_anomaly'] = False
mask_rc = window['is_row_count'] & window['p5_global'].notna()
window.loc[mask_rc, 'is_anomaly'] = window.loc[mask_rc, 'value'] <= window.loc[mask_rc, 'p5_global']
mask_de = window['is_duration_or_error'] & window['p95_global'].notna()
window.loc[mask_de, 'is_anomaly'] = window.loc[mask_de, 'value'] >= window.loc[mask_de, 'p95_global']

# 5) Select anomaly rows and prepare compact output
anomalies = window[window['is_anomaly']].copy()
# Round thresholds to 6 decimals (already rounded in global_thresh, but ensure)
anomalies['p95_global'] = anomalies['p95_global'].round(6)
anomalies['p5_global']  = anomalies['p5_global'].round(6)

# Select and rename columns, sort by timestamp ascending, limit to first 50 rows
anomalies_output = (
    anomalies[['timestamp','cmdb_id','trace_name','value','p95_global','p5_global']]
    .rename(columns={'timestamp':'timestamp_utc'})
    .sort_values('timestamp_utc', ascending=True)
    .head(50)
    .reset_index(drop=True)
)

anomalies_output

```
Out[7]:
```
```python
summary = (
    "Summary (trace anomalies for to_ts-travel-service / to_ts-travel2-service in 2024-01-24 03:17:05–03:47:05 UTC):\n\n"
    "- Many error_rate series targeting travel-service and travel2-service are flagged as anomalies starting at ~2024-01-24 03:20:00 UTC. "
    "Global p95 and p5 for these error_rate traces are 0.000000, so sampled values of 0.0 meet the anomaly rule used here and appear across multiple cmdb_id (root, ts-travel-service, ts-travel2-service, ts-admin-travel-service, etc.).\n\n"
    "- row_count anomalies are present for travel-related traces at the same general timeframe (examples at 03:20:00 UTC): ts-travel-service.row_count had values like 259 (p95_global ~342.6, p5_global ~259.6) and ts-travel2-service.row_count had values like 531 (p95_global ~640.2, p5_global ~533.4). These produce a few points where row_count <= p5_global (low-count anomalies).\n\n"
    "- The query returned the first 50 anomaly rows (sorted by timestamp); the earliest anomalies are at 03:20:00 UTC and anomalies continue for several minutes afterward (many error_rate hits and occasional row_count lows).\n\n"
    "Actionable notes: investigate service logs/traces and upstream dependencies around 03:20:00 UTC for system-wide error signals, and inspect travel-related traffic/ingestion around 03:20–03:30 UTC for the observed row_count drops. Also verify whether zero-valued error_rate thresholds reflect normalization artifacts before treating them as definitive errors."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

timestamp_utc                  cmdb_id                              trace_name  value  p95_global  p5_global
0  2024-01-24 03:20:00+00:00                     root   trace.to_ts-travel-service.error_rate    0.0         0.0        0.0
1  2024-01-24 03:20:00+00:00       ts-travel2-service   trace.to_ts-travel2-service.row_count  531.0       640.2      533.4
2  2024-01-24 03:20:00+00:00       ts-travel2-service  trace.to_ts-travel2-service.error_rate    0.0         0.0        0.0
3  2024-01-24 03:20:00+00:00        ts-travel-service    trace.to_ts-travel-service.row_count  259.0       342.6      259.6
4  2024-01-24 03:20:00+00:00        ts-travel-service   trace.to_ts-travel-service.error_rate    0.0         0.0        0.0
..                       ...                      ...                                     ...    ...         ...        ...
45 2024-01-24 03:24:00+00:00       ts-travel2-service  trace.to_ts-travel2-service.error_rate    0.0         0.0        0.0
46 2024-01-24 03:24:00+00:00        ts-travel-service   trace.to_ts-travel-service.error_rate    0.0         0.0        0.0
47 2024-01-24 03:24:00+00:00          ts-food-service   trace.to_ts-travel-service.error_rate    0.0         0.0        0.0
48 2024-01-24 03:24:00+00:00                     root   trace.to_ts-travel-service.error_rate    0.0         0.0        0.0
49 2024-01-24 03:24:00+00:00  ts-admin-travel-service   trace.to_ts-travel-service.error_rate    0.0         0.0        0.0

[50 rows x 6 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[8]:

import pandas as pd
import numpy as np

# Reuse trace_df if available, otherwise load and parse timestamps as UTC
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")
trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

# 1) Compute global P95 per (cmdb_id, trace_name) using the full series (before filtering)
def p95(x):
    arr = x.dropna().values
    return float(np.percentile(arr, 95)) if arr.size>0 else np.nan

global_p95 = (
    trace_df.groupby(['cmdb_id','trace_name'], dropna=False)['value']
    .agg(p95_global=p95)
    .reset_index()
)
global_p95['p95_global'] = global_p95['p95_global'].round(6)

# 2) Filter to incident window and selected trace_name patterns
start_ts = pd.Timestamp("2024-01-24 03:17:05", tz='UTC')
end_ts   = pd.Timestamp("2024-01-24 03:47:05", tz='UTC')

tw = trace_df[(trace_df['timestamp'] >= start_ts) & (trace_df['timestamp'] <= end_ts)].copy()

substrs = [
    'to_ts-auth-service','to_ts-order-service','to_ts-route-service',
    'to_ts-train-service','to_ts-travel-service','to_ts-travel2-service'
]
# keep only trace_name that contains any of substrs AND endswith duration_mean or duration_p95
mask_sub = tw['trace_name'].str.contains("|".join(substrs), na=False)
mask_end = tw['trace_name'].str.endswith(('duration_mean','duration_p95'), na=False)
tw = tw[mask_sub & mask_end].copy()

# 3) Merge global p95 thresholds into windowed rows
tw = tw.merge(global_p95, on=['cmdb_id','trace_name'], how='left')

# 4) For each (cmdb_id, trace_name) present in window compute required summary and longest consecutive anomaly segment
rows = []
grouped = tw.groupby(['cmdb_id','trace_name'], sort=False)

for (cmdb_id, trace_name), g in grouped:
    g = g.sort_values('timestamp').reset_index(drop=True)
    points_in_window = int(g.shape[0])
    p95_global = float(g['p95_global'].iloc[0]) if not pd.isna(g['p95_global'].iloc[0]) else np.nan

    # Determine anomalies (value >= p95_global). If p95_global is NaN -> no anomalies
    if pd.isna(p95_global):
        g['is_anom'] = False
    else:
        g['is_anom'] = g['value'] >= p95_global

    anomaly_count = int(g['is_anom'].sum())
    earliest_anom_ts = g.loc[g['is_anom'], 'timestamp'].min() if anomaly_count>0 else pd.NaT

    # Compute longest consecutive anomaly segment
    longest_len = 0
    longest_start = pd.NaT
    longest_end = pd.NaT
    max_value_in_longest = np.nan

    if anomaly_count > 0:
        # identify consecutive groups
        grp_id = (g['is_anom'] != g['is_anom'].shift(1, fill_value=False)).cumsum()
        g['grp'] = grp_id
        anom_groups = g[g['is_anom']].groupby('grp', sort=False)
        for _, sub in anom_groups:
            length = int(sub.shape[0])
            if length > longest_len:
                longest_len = length
                longest_start = sub['timestamp'].min()
                longest_end = sub['timestamp'].max()
                max_value_in_longest = float(sub['value'].max())
            # if equal length, keep the earlier one (already satisfied because of iteration order)

    # prepare row
    rows.append({
        'cmdb_id': cmdb_id,
        'trace_name': trace_name,
        'p95_global': round(p95_global,6) if not pd.isna(p95_global) else np.nan,
        'points_in_window': points_in_window,
        'anomaly_count': anomaly_count,
        'earliest_anomaly_timestamp_utc': earliest_anom_ts,
        'longest_consecutive_anomaly_length': longest_len,
        'longest_segment_start_utc': longest_start,
        'longest_segment_end_utc': longest_end,
        'max_value_in_longest_segment': (round(max_value_in_longest,6) if not pd.isna(max_value_in_longest) else np.nan)
    })

# 5) Build summary DataFrame, sort and limit to top 20 rows
summary_df = pd.DataFrame(rows)
summary_df = summary_df.sort_values(['longest_consecutive_anomaly_length','anomaly_count'], ascending=[False, False]).head(20).reset_index(drop=True)

# Final compact output
summary_df

```
Out[8]:
```
Summary (duration_mean / duration_p95 traces for services to auth/order/route/train/travel/travel2 during 2024-01-24 03:17:05–03:47:05 UTC):

Key points
- Most affected trace series show only short, brief latency anomalies: anomaly_count is typically 2 and the longest consecutive anomaly segment length is 1–2 points (minutes).
- The longest consecutive anomaly segments (length = 2) are:
  - root | trace.to_ts-travel-service.duration_p95 — p95_global=0.136866, points=25, anomaly_count=2, earliest=2024-01-24 03:28:00+00:00; longest segment 03:28–03:29, max_in_segment=0.210923
  - ts-auth-service | trace.to_ts-auth-service.duration_mean — p95_global=0.003946, points=25, anomaly_count=2, earliest=2024-01-24 03:29:00+00:00; longest segment 03:29–03:30, max_in_segment=0.004143
  - ts-train-service | trace.to_ts-train-service.duration_mean — p95_global=0.002253, points=25, anomaly_count=2, earliest=2024-01-24 03:37:00+00:00; longest segment 03:37–03:38, max_in_segment=0.002925

Other observations
- Many trace.duration_mean and duration_p95 series have anomaly_count = 2 (a few have 1) with earliest anomalies spread between ~03:20 and ~03:39 UTC depending on series.
- Several root and service-level travel traces show single-minute spikes (longest length = 1) at various times (e.g., 03:21, 03:22, 03:24, 03:32).
- No long-lived (>2 consecutive minutes) latency degradation was observed in these selected duration metrics within the window.

Interpretation / next steps
- The latency issues are short-lived and scattered; investigate traces and logs around the listed earliest anomaly timestamps (notably ~03:28–03:30 for travel/auth and ~03:37 for train) to find correlated errors or upstream events.
- Because anomalies are brief, check request patterns, load spikes, or transient dependency latencies at those minutes rather than a sustained service outage.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                 trace_name  p95_global  points_in_window  anomaly_count earliest_anomaly_timestamp_utc  longest_consecutive_anomaly_length longest_segment_start_utc   longest_segment_end_utc  max_value_in_longest_segment
0                        root    trace.to_ts-travel-service.duration_p95    0.136866                25              2      2024-01-24 03:28:00+00:00                                   2 2024-01-24 03:28:00+00:00 2024-01-24 03:29:00+00:00                      0.210923
1             ts-auth-service     trace.to_ts-auth-service.duration_mean    0.003946                25              2      2024-01-24 03:29:00+00:00                                   2 2024-01-24 03:29:00+00:00 2024-01-24 03:30:00+00:00                      0.004143
2            ts-train-service    trace.to_ts-train-service.duration_mean    0.002253                25              2      2024-01-24 03:37:00+00:00                                   2 2024-01-24 03:37:00+00:00 2024-01-24 03:38:00+00:00                      0.002925
3                        root     trace.to_ts-auth-service.duration_mean    0.419228                25              2      2024-01-24 03:24:00+00:00                                   1 2024-01-24 03:24:00+00:00 2024-01-24 03:24:00+00:00                      0.432424
4                        root      trace.to_ts-auth-service.duration_p95    1.112707                25              2      2024-01-24 03:24:00+00:00                                   1 2024-01-24 03:24:00+00:00 2024-01-24 03:24:00+00:00                      1.122620
5                        root    trace.to_ts-order-service.duration_mean    0.484948                24              2      2024-01-24 03:39:00+00:00                                   1 2024-01-24 03:39:00+00:00 2024-01-24 03:39:00+00:00                      0.488244
6                        root     trace.to_ts-order-service.duration_p95    0.623377                24              2      2024-01-24 03:39:00+00:00                                   1 2024-01-24 03:39:00+00:00 2024-01-24 03:39:00+00:00                      0.802653
7                        root   trace.to_ts-travel-service.duration_mean    0.076254                25              2      2024-01-24 03:21:00+00:00                                   1 2024-01-24 03:21:00+00:00 2024-01-24 03:21:00+00:00                      0.077678
8                        root  trace.to_ts-travel2-service.duration_mean    0.056527                25              2      2024-01-24 03:22:00+00:00                                   1 2024-01-24 03:22:00+00:00 2024-01-24 03:22:00+00:00                      0.056661
9                        root   trace.to_ts-travel2-service.duration_p95    0.104484                25              2      2024-01-24 03:25:00+00:00                                   1 2024-01-24 03:25:00+00:00 2024-01-24 03:25:00+00:00                      0.104946
10    ts-admin-travel-service   trace.to_ts-travel-service.duration_mean    0.122339                25              2      2024-01-24 03:20:00+00:00                                   1 2024-01-24 03:20:00+00:00 2024-01-24 03:20:00+00:00                      0.123082
11    ts-admin-travel-service    trace.to_ts-travel-service.duration_p95    0.242491                25              2      2024-01-24 03:23:00+00:00                                   1 2024-01-24 03:23:00+00:00 2024-01-24 03:23:00+00:00                      0.244013
12    ts-admin-travel-service  trace.to_ts-travel2-service.duration_mean    0.135607                25              2      2024-01-24 03:20:00+00:00                                   1 2024-01-24 03:20:00+00:00 2024-01-24 03:20:00+00:00                      0.141245
13    ts-admin-travel-service   trace.to_ts-travel2-service.duration_p95    0.253432                25              2      2024-01-24 03:20:00+00:00                                   1 2024-01-24 03:20:00+00:00 2024-01-24 03:20:00+00:00                      0.272544
14            ts-auth-service      trace.to_ts-auth-service.duration_p95    0.002781                25              2      2024-01-24 03:24:00+00:00                                   1 2024-01-24 03:24:00+00:00 2024-01-24 03:24:00+00:00                      0.002833
15            ts-food-service   trace.to_ts-travel-service.duration_mean    0.018974                24              2      2024-01-24 03:20:00+00:00                                   1 2024-01-24 03:20:00+00:00 2024-01-24 03:20:00+00:00                      0.019431
16            ts-food-service    trace.to_ts-travel-service.duration_p95    0.039909                24              2      2024-01-24 03:20:00+00:00                                   1 2024-01-24 03:20:00+00:00 2024-01-24 03:20:00+00:00                      0.041291
17  ts-inside-payment-service    trace.to_ts-order-service.duration_mean    0.210875                25              2      2024-01-24 03:32:00+00:00                                   1 2024-01-24 03:32:00+00:00 2024-01-24 03:32:00+00:00                      0.408550
18  ts-inside-payment-service     trace.to_ts-order-service.duration_p95    0.214788                25              2      2024-01-24 03:32:00+00:00                                   1 2024-01-24 03:32:00+00:00 2024-01-24 03:32:00+00:00                      1.112534
19           ts-order-service    trace.to_ts-order-service.duration_mean    0.223495                25              2      2024-01-24 03:32:00+00:00                                   1 2024-01-24 03:32:00+00:00 2024-01-24 03:32:00+00:00                      0.243402```
```